# Load and Clean CIC-IDS2017

Run each cell below in order (click the ▶ play button next to each cell, or press Shift+Enter).

If a cell errors with something like `ModuleNotFoundError: No module named 'pandas'`, run the install cell right below this one first.

In [1]:
# Only run this once if pandas/numpy aren't already installed - safe to skip if it's already there
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import glob
import numpy as np
import pandas as pd

## Set the folder path

This is the only thing you should need to change. Point it at wherever you extracted `MachineLearningCSV.zip`.

In [3]:
import os
CSV_FOLDER = os.path.expanduser("~/Downloads/MachineLearningCVE")
print(CSV_FOLDER)

/Users/augustinejoy/Downloads/MachineLearningCVE


In [4]:
import os
print(os.listdir(CSV_FOLDER))

['Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', 'Monday-WorkingHours.pcap_ISCX.csv', 'Friday-WorkingHours-Morning.pcap_ISCX.csv', 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', 'Tuesday-WorkingHours.pcap_ISCX.csv', 'Wednesday-workingHours.pcap_ISCX.csv', 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv']


## Load all 8 CSV files into one table

In [5]:
files = glob.glob(f"{CSV_FOLDER}/*.csv")
print(f"Found {len(files)} CSV files:")
for f in files:
    print(f"  {f}")

frames = [pd.read_csv(f, low_memory=False) for f in files]
raw = pd.concat(frames, ignore_index=True)
raw.columns = raw.columns.str.strip()  # column names have inconsistent leading/trailing spaces

print(f"\nTotal raw rows: {len(raw)}")

Found 8 CSV files:
  /Users/augustinejoy/Downloads/MachineLearningCVE/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  /Users/augustinejoy/Downloads/MachineLearningCVE/Monday-WorkingHours.pcap_ISCX.csv
  /Users/augustinejoy/Downloads/MachineLearningCVE/Friday-WorkingHours-Morning.pcap_ISCX.csv
  /Users/augustinejoy/Downloads/MachineLearningCVE/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  /Users/augustinejoy/Downloads/MachineLearningCVE/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  /Users/augustinejoy/Downloads/MachineLearningCVE/Tuesday-WorkingHours.pcap_ISCX.csv
  /Users/augustinejoy/Downloads/MachineLearningCVE/Wednesday-workingHours.pcap_ISCX.csv
  /Users/augustinejoy/Downloads/MachineLearningCVE/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv

Total raw rows: 2830743


## Look at the raw data

Useful for your "before" screenshot on the DataPrep_EDA tab.

In [8]:
raw.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,22,166,1,1,0,0,0,0,0.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
1,60148,83,1,2,0,0,0,0,0.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
2,123,99947,1,1,48,48,48,48,48.0,0.0,...,40,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
3,123,37017,1,1,48,48,48,48,48.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
4,0,111161336,147,0,0,0,0,0,0.0,0.0,...,0,1753752.625,2123197.578,4822992,95,9463032.7,2657727.996,13600000,5700287,BENIGN


In [9]:
raw['Label'].value_counts()

Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

## Clean the data

This handles two known issues in this dataset: some numeric columns contain infinity values (where a flow duration was 0), and there are duplicate rows across the files.

In [10]:
cleaned = raw.copy()

cleaned.replace([np.inf, -np.inf], np.nan, inplace=True)

before = len(cleaned)
cleaned.dropna(inplace=True)
print(f"Dropped {before - len(cleaned)} rows with missing/infinite values ({before} -> {len(cleaned)})")

before = len(cleaned)
cleaned.drop_duplicates(inplace=True)
print(f"Dropped {before - len(cleaned)} duplicate rows ({before} -> {len(cleaned)})")

Dropped 2867 rows with missing/infinite values (2830743 -> 2827876)
Dropped 307078 duplicate rows (2827876 -> 2520798)


In [11]:
cleaned['Label'].value_counts()


Label
BENIGN                        2095057
DoS Hulk                       172846
DDoS                           128014
PortScan                        90694
DoS GoldenEye                   10286
FTP-Patator                      5931
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1948
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

In [12]:
cleaned['Label'] = cleaned['Label'].str.replace('\ufffd', '-', regex=False)

## Look at the cleaned data

Useful for your "after" screenshot.

In [13]:
cleaned.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,22,166,1,1,0,0,0,0,0.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
1,60148,83,1,2,0,0,0,0,0.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
2,123,99947,1,1,48,48,48,48,48.0,0.0,...,40,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
3,123,37017,1,1,48,48,48,48,48.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
4,0,111161336,147,0,0,0,0,0,0.0,0.0,...,0,1753752.625,2123197.578,4822992,95,9463032.7,2657727.996,13600000,5700287,BENIGN


In [14]:
cleaned['Label'].value_counts()

Label
BENIGN                        2095057
DoS Hulk                       172846
DDoS                           128014
PortScan                        90694
DoS GoldenEye                   10286
FTP-Patator                      5931
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1948
Web Attack - Brute Force         1470
Web Attack - XSS                  652
Infiltration                       36
Web Attack - Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

## Save both versions

In [15]:
raw.sample(5000, random_state=1).to_csv("cicids2017_raw_sample.csv", index=False)
cleaned.to_csv("cicids2017_cleaned.csv", index=False)
print("Saved cicids2017_raw_sample.csv and cicids2017_cleaned.csv")

Saved cicids2017_raw_sample.csv and cicids2017_cleaned.csv
